# Notebook 5. CBOW Word Embeddings From Scratch (PyTorch)

*ML & NLP course - Data Trainers LLC - Axel Sirota*

## The story

In Module 1 our logistic-regression classifier saw words as **one-hot columns**: `king` and `queen` sat on orthogonal axes, `dog` was as far from `canine` as from `car`, and the model had no way to know that *good* and *great* belonged together. That is a ceiling on how much meaning any downstream classifier can recover from such a representation.

This notebook lifts that ceiling. We will learn **dense word vectors** directly from Yelp reviews, using a small neural network called **Continuous Bag of Words (CBOW)**. By the end you will have:

1. A 100-dimensional vector for every vocabulary word.
2. Confirmation that `vec('good')` is closer to `vec('great')` than to `vec('bad')`.
3. A PyTorch training loop you wrote yourself, which you will reuse in the remaining notebooks of this course.

## Learning objectives

By the end of this notebook you will be able to:

1. Explain the **distributional hypothesis**: *you shall know a word by the company it keeps* (Firth, 1957).
2. Justify why dense embeddings beat one-hot encoding.
3. Describe **CBOW** (predict center from context) vs. **Skip-gram** (predict context from center) conceptually.
4. Build `(context, target)` pairs from raw text given a window size.
5. Write a custom PyTorch `Dataset` and wrap it in a `DataLoader`.
6. Implement CBOW in ~15 lines of PyTorch (`nn.Embedding`, `nn.Linear`).
7. Write a PyTorch training loop manually (forward -> loss -> backward -> step -> zero_grad).
8. Extract the embedding matrix, convert to gensim `KeyedVectors`, and query similar words.
9. Visualize embeddings in 2D with PCA.

## Prerequisites

- Notebooks 1-4 (text preprocessing, classification, similarity intuition).
- Basic PyTorch familiarity (tensors, `.to(device)`, `nn.Module`). If you need a refresher, pause here and skim the official *60-minute blitz*.

## How to run

- **Colab (recommended):** Runtime -> Change runtime type -> GPU. Training takes ~2-3 min on a T4.
- **CPU:** Works but slower (5-10 min for 5 epochs).

## Section 0. Environment Setup

We install everything we need, import libraries, pin random seeds, and detect the GPU.

Reproducibility matters: with the same seed you should see the same loss curve we do.

In [ ]:
# Install required packages (run this first on Colab).
# If running locally in a virtualenv that already has these, you can skip.

!pip install -q torch gensim textblob scikit-learn matplotlib pandas numpy
!python -m textblob.download_corpora

In [ ]:
# Core imports - grouped visualization -> data -> model -> training
import random
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from textblob import TextBlob
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# ---- Hyperparameters (single source of truth) ----
SEED = 42
CORPUS_SIZE = 5000      # number of Yelp reviews to train on
VOCAB_MIN_FREQ = 5      # drop words that appear fewer than this many times
WINDOW_SIZE = 2         # context = [w-2, w-1, w+1, w+2]
EMBEDDING_DIM = 100     # dimensionality of each word vector
BATCH_SIZE = 128
EPOCHS = 5
LR = 1e-3

# ---- Seeds everywhere (Python, NumPy, PyTorch) ----
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---- Device detection ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("\nEnvironment setup complete.")

## Section 1. Why Embeddings?

### The one-hot problem

In Module 1 we represented a word by its column index in a vocabulary. That means:

```python
king  = [1, 0, 0, 0, 0, ...]
queen = [0, 1, 0, 0, 0, ...]
dog   = [0, 0, 1, 0, 0, ...]
```

The cosine similarity between **any** two different one-hot vectors is exactly **0**. The representation claims `king` is *as* unrelated to `queen` as it is to `dog`, which is obviously wrong. The next cell verifies this numerically.

In [ ]:
# Tiny demo: one-hot vectors are mutually orthogonal
vocab_toy = ['king', 'queen', 'dog', 'car']
V = len(vocab_toy)

# Build one-hot matrix (one word per row)
one_hot = np.eye(V)

# Compute cosine similarity between every pair
sims = cosine_similarity(one_hot)
sim_df = pd.DataFrame(sims, index=vocab_toy, columns=vocab_toy)
print("Cosine similarity of one-hot vectors:")
print(sim_df)
print("\nNote: every off-diagonal entry is 0. King = dog = car, as far as the model can tell.")

### The distributional hypothesis

> *"You shall know a word by the company it keeps."* - J. R. Firth, 1957.

Consider the blank in:

> "I drink ___ every morning."

Words that fit (*coffee, tea, water, juice*) share meaning-adjacent properties (drinkable, liquid, morning-appropriate). Their **contexts** overlap.

**CBOW** (Mikolov et al., 2013) turns this hypothesis into an algorithm:

- Slide a window of size *k* over the text.
- For every position, take the **context** (the *2k* surrounding words) and try to **predict the center word**.
- The only way the network can succeed is to learn a dense vector for each word such that similar contexts produce similar center-word predictions.

CBOW predicts *center from context*. Its sibling **Skip-gram** does the opposite: it predicts *context from center*. Both fall under *Word2Vec*.

## Section 2. Building the Dataset

We use the classic Yelp reviews dataset. We subsample 5,000 reviews to keep training under a few minutes.

Pipeline:

1. Load CSV from Dropbox.
2. Tokenize each review with TextBlob.
3. Build a vocabulary (drop words with `freq < 5`).
4. Encode every token as an integer id.
5. Slide the CBOW window to produce `(context, target)` integer pairs.
6. Wrap those pairs in a PyTorch `Dataset` + `DataLoader`.

In [ ]:
# Load Yelp reviews directly from Dropbox
URL = 'https://www.dropbox.com/s/xds4lua69b7okw8/yelp.csv?dl=1'
yelp = pd.read_csv(URL)
print(f"Full dataset: {len(yelp):,} reviews")

# Subsample for speed - reproducible because np.random is seeded
yelp_sample = yelp.sample(n=CORPUS_SIZE, random_state=SEED).reset_index(drop=True)
print(f"Subsampled for CBOW training: {len(yelp_sample):,} reviews")

yelp_sample[['stars', 'text']].head(3)

In [ ]:
# Demo: tokenize a review with TextBlob (same tokenizer we use for the full corpus)
example = yelp_sample.iloc[0]['text']
print("Raw review:")
print(example[:200], '...')

tokens_example = [w.lower() for w in TextBlob(example).words]
print(f"\nFirst 20 tokens: {tokens_example[:20]}")
print(f"Token count: {len(tokens_example)}")

In [ ]:
# Tokenize ALL reviews once. This takes ~20s on the 5K subsample.
def tokenize(text):
    """Lowercase tokens from TextBlob. Returns a list of strings."""
    return [w.lower() for w in TextBlob(str(text)).words]

tokenized_reviews = [tokenize(t) for t in yelp_sample['text']]
print(f"Tokenized {len(tokenized_reviews):,} reviews")
print(f"Avg tokens/review: {np.mean([len(r) for r in tokenized_reviews]):.1f}")

# Flatten for vocab counting
all_tokens = [tok for review in tokenized_reviews for tok in review]
print(f"Total tokens: {len(all_tokens):,}")

In [ ]:
# Demo: build a word -> id vocabulary with a minimum-frequency cutoff
def build_vocab(tokens, min_freq=5):
    """Return (word2id, id2word). Reserve 0 for <UNK>."""
    counts = Counter(tokens)
    # Keep only words with frequency >= min_freq
    kept = [w for w, c in counts.most_common() if c >= min_freq]
    word2id = {'<UNK>': 0}
    for w in kept:
        word2id[w] = len(word2id)
    id2word = {i: w for w, i in word2id.items()}
    return word2id, id2word

word2id, id2word = build_vocab(all_tokens, min_freq=VOCAB_MIN_FREQ)
vocab_size = len(word2id)
print(f"Vocabulary size (min_freq={VOCAB_MIN_FREQ}): {vocab_size:,}")
print(f"Sample mappings: {list(word2id.items())[:10]}")

In [ ]:
# Encode every token as its integer id (unknown -> 0).
UNK = word2id['<UNK>']

def encode(tokens):
    return [word2id.get(t, UNK) for t in tokens]

encoded_reviews = [encode(r) for r in tokenized_reviews]
print("Tokens:", tokenized_reviews[0][:10])
print("IDs:   ", encoded_reviews[0][:10])

### The CBOW window

With `WINDOW_SIZE = 2`, for each center word `w_t` we look at `w_{t-2}, w_{t-1}, w_{t+1}, w_{t+2}` and try to predict `w_t`:

```
  pizza is the [best] food around
   ^    ^       ^    ^    ^
  w-2  w-1      w+1  w+2
  context = [pizza, is, food, around]
  target  = best
```

We cannot form a full window at the start or end of a sentence (not enough context on one side), so those positions are dropped. The function below produces a flat list of `(context_ids, target_id)` pairs across the whole corpus.

In [ ]:
# Demo: turn one encoded review into (context, target) pairs
def make_context_target_pairs(ids, window=2):
    """Return a list of (context_ids, target_id) for positions with a full window on both sides."""
    pairs = []
    for i in range(window, len(ids) - window):
        context = ids[i - window:i] + ids[i + 1:i + window + 1]
        target = ids[i]
        pairs.append((context, target))
    return pairs

# Walk through a 10-token toy example so you can see what's happening
toy_ids = encoded_reviews[0][:10]
toy_words = [id2word.get(i, '<UNK>') for i in toy_ids]
print("Tokens:", toy_words)
print("IDs:   ", toy_ids)

toy_pairs = make_context_target_pairs(toy_ids, window=WINDOW_SIZE)
print("\n(context, target) pairs:")
for ctx, tgt in toy_pairs:
    print(f"  {[id2word[i] for i in ctx]}  ->  {id2word[tgt]}")

### Lab 1. Build the `CBOWDataset`

Now wrap the pair-generation logic into a PyTorch `Dataset`. Your dataset should:

1. In `__init__`, iterate over every encoded review, call `make_context_target_pairs`, and store all pairs in a list.
2. In `__len__`, return the number of pairs.
3. In `__getitem__(idx)`, return `(context_tensor, target_tensor)` where:
   - `context_tensor` is a `torch.LongTensor` of shape `(2 * window,)`.
   - `target_tensor` is a 0-d `torch.LongTensor` (just an integer class index).

Then wrap it in a `DataLoader` with `batch_size=BATCH_SIZE`, `shuffle=True`, and `num_workers=0` (important on Colab: higher values can hang).

After building it, sanity-check by fetching one batch and printing the shapes. You should see `(BATCH_SIZE, 4)` for context and `(BATCH_SIZE,)` for target.

In [ ]:
# Lab 1: implement CBOWDataset

class CBOWDataset(Dataset):
    def __init__(self, encoded_reviews, window=2):
        # 1. Collect every (context, target) pair from every review.
        #    Hint: call make_context_target_pairs(review, window) for each review and extend a list.
        self.pairs = None  # YOUR CODE
        self.window = window

    def __len__(self):
        # 2. Return the number of pairs.
        return None  # YOUR CODE

    def __getitem__(self, idx):
        # 3. Return (context_tensor, target_tensor).
        #    context_tensor: torch.long, shape (2 * window,)
        #    target_tensor:  torch.long, scalar (0-d)
        context, target = None, None  # YOUR CODE
        return context, target


# 4. Build the dataset and a DataLoader.
dataset = None  # YOUR CODE
loader = None   # YOUR CODE (batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# Sanity check - uncomment once implemented
# print(f"Total training pairs: {len(dataset):,}")
# ctx_batch, tgt_batch = next(iter(loader))
# print(f"Context batch shape: {ctx_batch.shape}  dtype={ctx_batch.dtype}")
# print(f"Target batch shape:  {tgt_batch.shape}  dtype={tgt_batch.dtype}")

## Section 3. The CBOW Model

### Architecture

Input: batch of context ids of shape `(B, 4)`, where 4 comes from `2 * window = 4`.

```
(B, 4)  --nn.Embedding(V, D)-->  (B, 4, D)
        --mean over dim=1 ------>  (B, D)        <-- the CBOW averaging step
        --nn.Linear(D, V) ------>  (B, V)        <-- logits over the vocabulary
```

Key points:

- `nn.Embedding(V, D)` is just a `(V, D)` lookup table. Given an integer `i`, it returns row `i` of the table.
- We **average** (not concatenate) the 4 context embeddings. Why? Order-invariance and fixed output size.
- We output **logits** (no softmax!). `CrossEntropyLoss` will apply log-softmax internally. Applying softmax in `forward` and then using `CrossEntropyLoss` is a classic bug: double softmax collapses gradients.

The next cell walks through the shapes with a dummy tensor before you implement the module.

In [ ]:
# Demo: walk through the shapes with a dummy Embedding + Linear
dummy_V = 100
dummy_D = 8
dummy_emb = nn.Embedding(dummy_V, dummy_D)
dummy_lin = nn.Linear(dummy_D, dummy_V)

# Pretend batch of 3 contexts, each with 4 tokens
fake_ctx = torch.randint(0, dummy_V, (3, 4))
print("Input ids shape:        ", fake_ctx.shape)

e = dummy_emb(fake_ctx)
print("After Embedding:         ", e.shape, "  # (B, 2*window, D)")

pooled = e.mean(dim=1)
print("After mean over dim=1:   ", pooled.shape, "  # (B, D)")

logits = dummy_lin(pooled)
print("After Linear:            ", logits.shape, "  # (B, V) - logits over vocabulary")

print("\nAnd the embedding weight matrix itself is shape:", dummy_emb.weight.shape)

### Lab 2. Implement the CBOW model

Fill in the two layers in `__init__` and the three lines in `forward`. The model is intentionally small; the two layers and three-line forward pass are the entire architecture.

In [ ]:
# Lab 2: implement CBOW

class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # 1. Embedding lookup table of size (vocab_size, embedding_dim)
        self.embedding = None  # YOUR CODE
        # 2. Linear projection back to vocab_size logits
        self.linear = None     # YOUR CODE

    def forward(self, context_ids):
        # context_ids shape: (B, 2*window)
        # 3. Look up embeddings -> (B, 2*window, D)
        emb = None  # YOUR CODE
        # 4. Mean-pool over the context dimension -> (B, D)
        pooled = None  # YOUR CODE
        # 5. Project to logits -> (B, V). Do NOT apply softmax here.
        logits = None  # YOUR CODE
        return logits


# Instantiate and move to device
model = None  # YOUR CODE (use CBOW(vocab_size, EMBEDDING_DIM).to(device))

# Sanity check - uncomment once implemented
# print(model)
# n_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {n_params:,}")
#
# # Run a dummy forward pass
# dummy = torch.randint(0, vocab_size, (2, 2 * WINDOW_SIZE), device=device)
# out = model(dummy)
# print(f"Output logits shape: {out.shape}  (expected: (2, {vocab_size}))")

## Section 4. The Training Loop

This is the canonical PyTorch training loop. You will write it by hand; these six lines cover most of day-to-day PyTorch.

```python
for epoch in range(EPOCHS):
    for x, y in loader:
        x, y = x.to(device), y.to(device)   # 1. move tensors to GPU
        optimizer.zero_grad()               # 2. clear stale gradients
        logits = model(x)                   # 3. forward pass
        loss = loss_fn(logits, y)           # 4. compute loss
        loss.backward()                     # 5. backprop
        optimizer.step()                    # 6. update weights
```

**Gradient accumulation note.** PyTorch **accumulates** gradients by default. If you skip `optimizer.zero_grad()`, your updates become the sum of the current and all previous batches' gradients, and training silently goes wrong. Always call `zero_grad` at the start of each step.

### Sanity check: the initial loss

Before training, a random model predicts each of `V` classes with probability `1/V`. Cross-entropy of that uniform distribution is `ln(V)`. If your initial loss differs wildly from `ln(vocab_size)`, your setup is wrong.

In [ ]:
# Demo: set up loss + optimizer and verify the initial-loss sanity check
loss_fn = nn.CrossEntropyLoss()
# Note: we reference `model` from the previous lab. If you haven't filled it in yet,
# that lab must be completed first for these cells to run.
# optimizer = torch.optim.Adam(model.parameters(), lr=LR)
#
# # Expected initial loss ~ ln(vocab_size)
# expected = np.log(vocab_size)
# print(f"Expected initial loss ~ ln({vocab_size}) = {expected:.3f}")
#
# # Pull one batch, run forward, inspect loss before any training
# ctx, tgt = next(iter(loader))
# ctx, tgt = ctx.to(device), tgt.to(device)
# with torch.no_grad():
#     init_logits = model(ctx)
#     init_loss = loss_fn(init_logits, tgt)
# print(f"Actual initial loss: {init_loss.item():.3f}")
print("Uncomment the block above once Labs 1 and 2 are complete.")

In [ ]:
# Demo: a single backward pass - inspect a gradient to prove autograd works
# (Requires Labs 1 and 2 to be complete.)
#
# optimizer.zero_grad()
# logits = model(ctx)
# loss = loss_fn(logits, tgt)
# loss.backward()
# print("Gradient norm on embedding matrix:",
#       model.embedding.weight.grad.norm().item())
# print("Gradient shape:", model.embedding.weight.grad.shape)
print("Uncomment the block above once Labs 1 and 2 are complete.")

### Lab 3. Write the training loop

Fill in the 6 lines of the loop. Track the average loss per epoch in a list so we can plot it afterward.

Hyperparameters (already defined): `EPOCHS = 5`, `LR = 1e-3`, `BATCH_SIZE = 128`.

After training, you should see the loss drop from ~ln(vocab_size) to something noticeably smaller. The absolute value depends on your vocab; what matters is the downward trend.

In [ ]:
# Lab 3: the training loop

# Rebuild a fresh optimizer for this lab so students can re-run this cell cleanly
optimizer = torch.optim.Adam(model.parameters(), lr=LR) if model is not None else None

epoch_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for context, target in loader:
        # 1. Move tensors to the right device
        context, target = None, None  # YOUR CODE

        # 2. Clear gradients from the previous step (DO NOT FORGET THIS)
        None  # YOUR CODE

        # 3. Forward pass -> logits of shape (B, V)
        logits = None  # YOUR CODE

        # 4. Compute the loss
        loss = None    # YOUR CODE

        # 5. Backpropagate
        None  # YOUR CODE

        # 6. Update the weights
        None  # YOUR CODE

        total_loss += loss.item()
        n_batches += 1

    avg = total_loss / max(n_batches, 1)
    epoch_losses.append(avg)
    print(f"Epoch {epoch + 1}/{EPOCHS} - avg loss: {avg:.4f}")

In [ ]:
# Demo: plot the loss curve
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Avg training loss')
plt.title('CBOW training loss')
plt.grid(True, alpha=0.3)
plt.show()

### How do I know training is working?

Three things to check:

1. **Loss is decreasing.** If it is flat or increasing, something is wrong (usually forgotten `zero_grad`, wrong LR, or logits-vs-softmax bug).
2. **No NaNs.** If `loss.item()` prints `nan`, you have exploding gradients. Lower the LR or add `torch.nn.utils.clip_grad_norm_`.
3. **Validation loss diverges from training loss** (beyond some epoch) = **overfitting**. With CBOW on 5K reviews we are unlikely to overfit hard, but larger models do.

The next cell saves the model so you do not have to re-train if the kernel restarts.

In [ ]:
# Save the trained weights. state_dict is the recommended checkpoint format.
torch.save(model.state_dict(), 'cbow.pt')
print("Saved cbow.pt")
# To restore later:
#   model = CBOW(vocab_size, EMBEDDING_DIM).to(device)
#   model.load_state_dict(torch.load('cbow.pt', map_location=device))

## Section 5. Using the Embeddings

Training is over. The **only** thing we care about now is the learned `embedding.weight`: the `(V, D)` matrix whose rows are word vectors.

To query similar words comfortably we'll wrap that matrix in gensim's `KeyedVectors`, the same object `Word2Vec.load` returns, so the API feels familiar.

In [ ]:
# Extract the embedding matrix and wrap it in a gensim KeyedVectors
from gensim.models import KeyedVectors

# 1. Pull the learned weights out of the model.
#    detach() removes autograd tracking, cpu() moves off GPU, numpy() converts dtype.
embedding_matrix = model.embedding.weight.detach().cpu().numpy()
print(f"Embedding matrix shape: {embedding_matrix.shape}  (vocab_size, embedding_dim)")

# 2. Build a KeyedVectors object that maps words -> vectors.
kv = KeyedVectors(vector_size=EMBEDDING_DIM)
words = [id2word[i] for i in range(vocab_size)]
kv.add_vectors(words, embedding_matrix)
print(f"KeyedVectors contains {len(kv):,} words")

# 3. Sanity query - the learned embedding space should already cluster obvious synonyms
print("\nWords closest to 'good':")
for w, sim in kv.most_similar('good', topn=5):
    print(f"  {w:20s}  cos={sim:.3f}")

### Lab 4. Probe the embedding space

Query `most_similar` for each of these 5 anchor words and print the top-5 neighbors:

- `good`, `bad`, `service`, `food`, `expensive`

Then discuss (in a comment) what you see. Typical observations:

- `good` and `bad` are often *close* to each other, because both appear in the same slot in sentences. Distributional similarity is not the same as sentiment similarity.
- Domain words (`service`, `food`) should bring up other restaurant-scoped words.
- `expensive` should attract `pricey`, `overpriced`, `cheap` (again, same slot).

In [ ]:
# Lab 4: query the embedding space
anchors = ['good', 'bad', 'service', 'food', 'expensive']

for anchor in anchors:
    # 1. Guard against OOV - skip anchors not in the vocab
    if None:  # YOUR CODE (condition: anchor not in kv)
        print(f"'{anchor}' not in vocab - skipping")
        continue
    # 2. Find top-5 most similar and print them
    neighbors = None  # YOUR CODE (use kv.most_similar)
    print(f"\nTop-5 neighbors of '{anchor}':")
    for w, s in neighbors:
        print(f"  {w:20s}  cos={s:.3f}")

# 3. In a comment below, write one sentence per anchor describing what you see.
# EXAMPLE: 'good' pulls in {'great', 'amazing', ...}.
# YOUR OBSERVATIONS:
#   - good:
#   - bad:
#   - service:
#   - food:
#   - expensive:

### The famous analogy: `king - man + woman ≈ queen`

Word2Vec's poster-child result. It is intrinsic to the geometry of the embedding space: semantic directions emerge.

**Disclaimer:** trained on 5K restaurant reviews, this analogy is very unlikely to work for our CBOW. *king, queen, man, woman* barely appear. It works on the full Wikipedia-trained GloVe (next notebook).

We include it here because *trying it and watching it fail* is the best way to understand why in NB6 we reach for pretrained embeddings.

In [ ]:
# Demo: the analogy - expect weak results on Yelp
def try_analogy(kv, pos, neg, topn=5):
    try:
        return kv.most_similar(positive=pos, negative=neg, topn=topn)
    except KeyError as e:
        return f"OOV word in analogy: {e}"

print("king - man + woman ≈ ?")
print(try_analogy(kv, pos=['king', 'woman'], neg=['man']))

print("\ngood - great + bad ≈ ?  (domain analogy - more likely to work on Yelp)")
print(try_analogy(kv, pos=['good', 'bad'], neg=['great']))

### PCA: shrinking 100D to 2D for a scatter plot

We cannot visualize 100-dimensional vectors directly. **PCA** gives us the 2-D projection that preserves the most variance. It won't be faithful (a lot of structure lives in the lost 98 dimensions), but it's enough to eyeball clusters.

In [ ]:
# Demo: PCA the 50 most frequent vocabulary words down to 2D and plot
freq_order = [id2word[i] for i in range(1, 51)]  # skip <UNK> at id 0
freq_vecs = np.stack([kv[w] for w in freq_order])

pca_2d = PCA(n_components=2, random_state=SEED).fit_transform(freq_vecs)

plt.figure(figsize=(10, 7))
plt.scatter(pca_2d[:, 0], pca_2d[:, 1], s=30, alpha=0.6)
for (x, y), w in zip(pca_2d, freq_order):
    plt.text(x + 0.01, y + 0.01, w, fontsize=9)
plt.title('PCA of 50 most frequent words (CBOW-Yelp 100D -> 2D)')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.grid(True, alpha=0.3)
plt.show()

### Lab 5. Domain cluster plot

Pick 15-20 vocabulary words spanning three domains, for example:

- **food**: pizza, chicken, salad, burger, sandwich, taco, pasta
- **service**: waiter, staff, service, manager, host, waitress
- **ambience**: music, decor, atmosphere, seating, patio, lighting

Extract their vectors, PCA to 2D, and plot. Color each point by domain. Do domains form visible clusters?

Some words may be OOV, so filter them out before plotting.

In [ ]:
# Lab 5: domain cluster plot

# 1. Define your word groups. Feel free to tweak.
groups = {
    'food':     ['pizza', 'chicken', 'salad', 'burger', 'sandwich', 'taco', 'pasta'],
    'service':  ['waiter', 'staff', 'service', 'manager', 'host', 'waitress'],
    'ambience': ['music', 'decor', 'atmosphere', 'seating', 'patio', 'lighting'],
}

# 2. Filter to words that exist in the vocab.
filtered = None  # YOUR CODE: dict[str, list[str]] - only keep words where w in kv

# 3. Build a flat list of (word, domain) and their vectors.
flat_words = None   # YOUR CODE
flat_domains = None  # YOUR CODE
flat_vecs = None    # YOUR CODE: np.stack of kv[w] for each w

# 4. PCA down to 2D
coords_2d = None  # YOUR CODE

# 5. Scatter with a different color per domain
# domain_colors = {'food': 'tab:red', 'service': 'tab:blue', 'ambience': 'tab:green'}
# plt.figure(figsize=(10, 7))
# for d, color in domain_colors.items():
#     mask = [dom == d for dom in flat_domains]
#     plt.scatter(coords_2d[mask, 0], coords_2d[mask, 1], c=color, label=d, s=60)
# for (x, y), w in zip(coords_2d, flat_words):
#     plt.text(x + 0.01, y + 0.01, w, fontsize=9)
# plt.legend(); plt.title('Domain clusters in CBOW-Yelp embedding space')
# plt.xlabel('PC1'); plt.ylabel('PC2'); plt.grid(True, alpha=0.3)
# plt.show()

# 6. Observation: do the domains separate visibly? Write one sentence.
# YOUR OBSERVATION:

### Limitations of our tiny CBOW

We trained on 5,000 restaurant reviews for 5 epochs. That is a **toy** corpus. Expect:

- **Small, domain-skewed vocabulary.** `pizza` has a great vector, `parliament` does not exist at all.
- **Noisy similarities.** Low-frequency words don't get enough signal, so their vectors drift.
- **No famous analogies** (`king - man + woman`). Those need billions of tokens of general-purpose text.
- **Shallow semantics.** CBOW learns distributional similarity and cannot tell synonyms from antonyms (both appear in the same context slot).

## Section 6. Wrap-up

### What we built vs. the real Word2Vec paper

Our implementation differs from Mikolov et al. (2013) in two ways:

1. **Loss.** We use full cross-entropy over the whole vocabulary. The original paper uses **negative sampling** or **hierarchical softmax** because computing softmax over a million-word vocabulary is expensive. For our 5K-review corpus with ~3K vocab, the full softmax is fine.
2. **Subsampling & phrase detection.** The paper downsamples frequent words (`the`, `a`) and detects multi-word phrases (`New_York`). We skipped both for simplicity.

The next cell compares our from-scratch CBOW to gensim's production `Word2Vec` on the same corpus.

In [ ]:
# Demo: gensim Word2Vec as a reference point
import time
from gensim.models import Word2Vec

t0 = time.time()
w2v_ref = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=EMBEDDING_DIM,
    window=WINDOW_SIZE,
    min_count=VOCAB_MIN_FREQ,
    sg=0,            # sg=0 => CBOW, sg=1 => Skip-gram
    workers=2,
    epochs=EPOCHS,
    seed=SEED,
)
t_gensim = time.time() - t0
print(f"gensim Word2Vec trained in {t_gensim:.1f}s")

print("\nOur CBOW 'good':")
for w, s in kv.most_similar('good', topn=5):
    print(f"  {w:20s}  cos={s:.3f}")

print("\ngensim 'good':")
for w, s in w2v_ref.wv.most_similar('good', topn=5):
    print(f"  {w:20s}  cos={s:.3f}")

### Optional lab. Sweep embedding_dim

Re-train CBOW with `embedding_dim ∈ {50, 100, 300}` and compare the `most_similar('food')` lists. Bigger is not always better on a small corpus: at 300D with only ~3K vocab you may overfit.

Skip this if time is tight.

In [ ]:
# Optional lab: dim sweep - only run if you have a few extra minutes

def train_cbow_quick(dim, epochs=EPOCHS):
    # 1. Rebuild model with new dim
    m = None  # YOUR CODE
    # 2. Fresh optimizer
    opt = None  # YOUR CODE
    # 3. Train loop (same 6 lines as Lab 3)
    for ep in range(epochs):
        m.train()
        for ctx, tgt in loader:
            ctx, tgt = ctx.to(device), tgt.to(device)
            opt.zero_grad()
            logits = m(ctx)
            loss = loss_fn(logits, tgt)
            loss.backward()
            opt.step()
    return m

# Uncomment to run the sweep
# for dim in [50, 100, 300]:
#     m = train_cbow_quick(dim)
#     mat = m.embedding.weight.detach().cpu().numpy()
#     kv_d = KeyedVectors(vector_size=dim)
#     kv_d.add_vectors(words, mat)
#     print(f"\n--- dim={dim} --- most similar to 'food':")
#     for w, s in kv_d.most_similar('food', topn=5):
#         print(f"  {w:20s}  cos={s:.3f}")

## What you learned

- **Distributional hypothesis**: meaning lives in context.
- **CBOW**: one of the two Word2Vec flavors. Predict the center word from its context.
- **PyTorch workflow**: `Dataset`, `DataLoader`, `nn.Module`, `CrossEntropyLoss`, `Adam`, and the 6-line training loop.
- **Dense embeddings**: extracted from `model.embedding.weight`, usable via `KeyedVectors`.
- **Limitations of small corpora**: why we move to pretrained GloVe in NB6.

### Self-check quiz

1. Why is mean-pooling the context embeddings (instead of concatenating them) a reasonable choice? What is the trade-off?
2. What would the initial loss be if `vocab_size = 10000`?
3. Your `most_similar('good')` returns `['great', 'amazing', 'excellent', ..., 'terrible']`. Explain how `terrible` can appear in the list even though it's an antonym.

### Next up

**Notebook 6. Pretrained Embeddings (GloVe).** We reuse the CBOW class but initialize `nn.Embedding` from Stanford's GloVe 6B vectors, then compare frozen vs. fine-tuned. You'll see `king - man + woman ≈ queen` finally work.